In [ ]:
# Imports and constants
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    precision_recall_curve,
    auc,
    cohen_kappa_score,
    balanced_accuracy_score
)

EPS = 1e-8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 5
LR = 1e-4
epochs = 50

In [ ]:
# Model 

class DemographicEmbedding(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, embed_dim),
            nn.ReLU()
        )

    def forward(self, age, sex):
        x = torch.stack([age, sex], dim=1)
        return self.net(x)


class BayesianSkip(nn.Module):
    def __init__(self, channels, demo_dim=32):
        super().__init__()
        self.mu = nn.Conv3d(channels + demo_dim, channels, 1)
        self.logvar = nn.Conv3d(channels + demo_dim, channels, 1)

    def forward(self, x, demo):
        B, C, D, H, W = x.shape

        demo = demo.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        demo = demo.expand(B, demo.shape[1], D, H, W)

        x = torch.cat([x, demo], dim=1)

        mu = self.mu(x)
        logvar = torch.clamp(self.logvar(x), -10, -2)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        out = mu + eps * std
        return torch.nan_to_num(out, nan=0.0)



class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim

        self.conv1 = nn.Conv3d(1, 32, 3, 2, 1)
        self.conv2 = nn.Conv3d(32, 64, 3, 2, 1)
        self.conv3 = nn.Conv3d(64, 128, 3, 2, 1)
        self.conv4 = nn.Conv3d(128, 256, 3, 2, 1)

        self.bn1 = nn.BatchNorm3d(32)
        self.bn2 = nn.BatchNorm3d(64)
        self.bn3 = nn.BatchNorm3d(128)
        self.bn4 = nn.BatchNorm3d(256)

        self.flatten_dim = None
        self.mu = None
        self.logvar = None

    def forward(self, x):
        B = x.size(0)

        s1 = F.relu(self.bn1(self.conv1(x)))
        s2 = F.relu(self.bn2(self.conv2(s1)))
        s3 = F.relu(self.bn3(self.conv3(s2)))
        s4 = F.relu(self.bn4(self.conv4(s3)))

        flat = s4.view(B, -1)

        if self.flatten_dim is None:
            self.flatten_dim = flat.shape[1]
            self.mu = nn.Linear(self.flatten_dim, self.latent_dim).to(x.device)
            self.logvar = nn.Linear(self.flatten_dim, self.latent_dim).to(x.device)

        mu = self.mu(flat)
        logvar = torch.clamp(self.logvar(flat), -10, -2)

        return mu, logvar, (s1, s2, s3), s4



class Decoder(nn.Module):
    def __init__(self, latent_dim, demo_dim=32):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 256 * 10 * 12 * 10)

        self.bskip1 = BayesianSkip(128, demo_dim)
        self.bskip2 = BayesianSkip(64, demo_dim)
        self.bskip3 = BayesianSkip(32, demo_dim)

        self.deconv1 = nn.ConvTranspose3d(256, 128, 3, 2, 1, 1)
        self.deconv2 = nn.ConvTranspose3d(128, 64, 3, 2, 1, 1)
        self.deconv3 = nn.ConvTranspose3d(64, 32, 3, 2, 1, 1)
        self.deconv4 = nn.ConvTranspose3d(32, 1, 3, 2, 1, 1)

        self.bn1 = nn.BatchNorm3d(128)
        self.bn2 = nn.BatchNorm3d(64)
        self.bn3 = nn.BatchNorm3d(32)

    def forward(self, z, skips, demo):
        s1, s2, s3 = skips

        x = self.fc(z)
        x = x.view(-1, 256, 10, 12, 10)

        x = F.relu(self.bn1(self.deconv1(x))) + self.bskip1(s3, demo)
        x = F.relu(self.bn2(self.deconv2(x))) + self.bskip2(s2, demo)
        x = F.relu(self.bn3(self.deconv3(x))) + self.bskip3(s1, demo)

        x = self.deconv4(x)
        return torch.sigmoid(x)


def global_bottleneck_pooling(bottleneck):
    pooled = F.adaptive_avg_pool3d(bottleneck, 1)
    return pooled.view(pooled.size(0), -1)


class CDRHead(nn.Module):
    def __init__(self, latent_dim, skip_dim=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(latent_dim + skip_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, z, skip_feat):
        x = torch.cat([z, skip_feat], dim=1)
        return self.net(x)



class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()

        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.demo_embed = DemographicEmbedding()
        self.cdr_head = CDRHead(latent_dim, skip_dim=256)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, age, sex):

        mu, logvar, skips, bottleneck = self.encoder(x)

        z = self.reparameterize(mu, logvar)

        demo = self.demo_embed(age, sex)

        recon = self.decoder(z, skips, demo)

        skip_feat = global_bottleneck_pooling(bottleneck)

        cdr_pred = self.cdr_head(z, skip_feat)

        return {
            "recon": recon,
            "cdr_pred": cdr_pred,
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "bottleneck": bottleneck
        }

In [ ]:
# Data preparation

class MRIDataset(torch.utils.data.Dataset):
    def __init__(self, paths, ages, sexes, cdrs=None, groups=None):
        self.paths = paths
        self.ages = ages
        self.sexes = sexes
        self.cdrs = cdrs
        self.groups = groups

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        volume = np.load(self.paths[idx]).astype(np.float32)

        # 🔥 normalize per-sample
        vmin, vmax = volume.min(), volume.max()
        if vmax - vmin > EPS:
            volume = (volume - vmin) / (vmax - vmin)
        else:
            volume = np.zeros_like(volume)

        volume = torch.tensor(volume, dtype=torch.float32)

        age = torch.tensor(self.ages[idx], dtype=torch.float32)
        sex = torch.tensor(self.sexes[idx], dtype=torch.float32)

        outputs = [volume, age, sex]

        if self.cdrs is not None:
            cdr = torch.tensor(self.cdrs[idx], dtype=torch.float32)
            outputs.append(cdr)

        if self.groups is not None:
            group = torch.tensor(self.groups[idx], dtype=torch.long)
            outputs.append(group)

        return tuple(outputs)


def split_tables(df):
    # Table 1: progression (0 → >0)
    has_zero = df[df["CDRSUM"] == 0].groupby("OASIS3_id").size()
    has_nonzero = df[df["CDRSUM"] > 0].groupby("OASIS3_id").size()

    valid_ids = set(has_zero.index).intersection(set(has_nonzero.index))

    table1_rows = []
    for oid in valid_ids:
        sub = df[df["OASIS3_id"] == oid]

        row_zero = sub[sub["CDRSUM"] == 0].iloc[0]
        row_max = sub.loc[sub["CDRSUM"].idxmax()]

        table1_rows.append(row_zero)
        table1_rows.append(row_max)

    table1 = pd.DataFrame(table1_rows)

    # Table 2: max CDR per id
    idx = df.groupby("OASIS3_id")["CDRSUM"].idxmax()
    idx = idx.dropna()

    table2 = df.loc[idx]

    # Table 3: everything else
    used_idx = set(table1.index).union(set(table2.index))
    table3 = df.drop(index=used_idx, errors="ignore")

    return table1, table2, table3


def severity_group(cdr):
    if cdr < 0.5:
        return 0
    elif 0 < cdr <= 4.0:
        return 1
    elif 4.0 < cdr <= 9.0:
        return 2
    elif 9.0 < cdr <= 15.5:
        return 3
    else:
        return 4
        

def get_cdr_distribution(df):
    counts = df['severity_group'].value_counts().sort_index()
    proportions = df['severity_group'].value_counts(normalize=True).sort_index()

    print("CDR Counts:\n", counts)
    print("\nCDR Proportions:\n", proportions)

    return counts, proportions



def stratified_train_test_split(df, test_size=0.2, random_state=42):
    """
    Stratified split based on severity_group with safeguards for rare classes.
    """

    group_counts = df['severity_group'].value_counts()
    print("Group counts:\n", group_counts)

    rare_groups = group_counts[group_counts < 5].index.tolist()

    if len(rare_groups) > 0:
        print(f"\n Rare groups detected (will ALL go to training): {rare_groups}")

        rare_df = df[df['severity_group'].isin(rare_groups)]
        common_df = df[~df['severity_group'].isin(rare_groups)]

        # Stratified split only on common groups
        train_common, test_common = train_test_split(
            common_df,
            test_size=test_size,
            stratify=common_df['severity_group'],
            random_state=random_state
        )

        # Add rare samples ONLY to training
        train_df = pd.concat([train_common, rare_df]).reset_index(drop=True)
        test_df = test_common.reset_index(drop=True)

    else:
        # Standard stratified split
        train_df, test_df = train_test_split(
            df,
            test_size=test_size,
            stratify=df['severity_group'],
            random_state=random_state
        )

    # Verify distributions
    print("\nTrain distribution:")
    print(train_df['severity_group'].value_counts(normalize=True).sort_index())

    print("\nTest distribution:")
    print(test_df['severity_group'].value_counts(normalize=True).sort_index())

    return train_df, test_df

In [ ]:
# import table with dempgraphics, cdr scores, and paths to cleaned data

# Data for T1W MRIs
mri_t1 = pd.read_csv('YOUR_PATH')

progressions_t1, maxcdrs_t1, unused_t1 =split_tables(mri_t1)

train_df_t1, intermed_t1 = stratified_train_test_split(maxcdrs_t1)

unused_t1 = unused_t1[unused_t1["OASIS3_id"].isin(intermed_t1["OASIS3_id"])]

test_df_t1 = pd.concat([intermed_t1, unused_t1w], ignore_index=True)

test_df_t1['severity_group'] = test_df_t1['CDRSUM'].apply(severity_group)

train_d_t1 = MRIDataset(
    paths=train_df_t1["fixed_path"].to_list(),
    ages=train_df_t1["age_at_session"].to_list(),
    sexes=train_df_t1["sex"].to_list(),
    cdrs = train_df_t1['CDRSUM'].to_list(),
    groups = train_df_t1['severity_group'].to_list()
)

train_loader_t1 = DataLoader(
    train_d_t1,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
test_d_t1 = MRIDataset(
    paths=test_df_t1["fixed_path"].to_list(),
    ages=test_df_t1["age_at_session"].to_list(),
    sexes=test_df_t1["sex"].to_list(),
    cdrs = test_df_t1['CDRSUM'].to_list(),
    groups = test_df_t1['severity_group'].to_list()
)

test_loader_t1 = DataLoader(
    test_d_t1,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

# Data for T2W MRIs
mri_t2 = pd.read_csv('YOUR_PATH')

progressions_t2, maxcdrs_t2, unused_t2 = split_tables(mri_t2)

train_df_t2, intermed_t2 = stratified_train_test_split(maxcdrs_t2)

unused_t2 = unused_t2[unused_t2["OASIS3_id"].isin(intermed_t2["OASIS3_id"])]

test_df_t2 = pd.concat([intermed_t2, unused_t2w], ignore_index=True)

test_df_t2['severity_group'] = test_df_t2['CDRSUM'].apply(severity_group)

train_d_t2 = MRIDataset(
    paths=train_df_t2["fixed_path"].to_list(),
    ages=train_df_t2["age_at_session"].to_list(),
    sexes=train_df_t2["sex"].to_list(),
    cdrs = train_df_t2['CDRSUM'].to_list(),
    groups = train_df_t2['severity_group'].to_list()
)

train_loader_t2 = DataLoader(
    train_d_t2,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
test_d_t2  = MRIDataset(
    paths=test_df_t2["fixed_path"].to_list(),
    ages=test_df_t2["age_at_session"].to_list(),
    sexes=test_df_t2["sex"].to_list(),
    cdrs = test_df_t2['CDRSUM'].to_list(),
    groups = test_df_t2['severity_group'].to_list()
)

test_loader_t2 = DataLoader(
    test_d_t2,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [ ]:
# Train function

def info_nce_loss(feat, temperature=0.1):

    feat = F.normalize(feat, dim=1)

    sim = torch.mm(feat, feat.t())  
    sim = sim / temperature

    B = feat.size(0)
    labels = torch.arange(B, device=feat.device)

    loss = F.cross_entropy(sim, labels)

    return loss

def train_one_epoch(model, loader, optimizer, beta= 1, gamma=0.2):

    model.train()

    total_loss = 0.0
    total_rec = 0.0
    total_kl = 0.0
    total_cdr = 0.0
    total_contrast = 0.0

    valid_batches = 0

    for batch in tqdm(loader, desc="Training"):

        x, age, sex, cdr, _ = batch

        x = x.to(device)
        x = torch.clamp(x, min=0)
        x = x / (x.max() + 1e-5)
        age = age.to(device)
        sex = sex.to(device)
        cdr = cdr.to(device)

        optimizer.zero_grad()

        out = model(x, age, sex)

        recon = out["recon"]
        L_rec = F.l1_loss(recon, x)

        mu = out["mu"]
        logvar = out["logvar"]

        L_KL = -0.5 * torch.mean(
            1 + logvar - mu.pow(2) - logvar.exp()
        )

        cdr_pred = out["cdr_pred"].squeeze(-1)
        cdr_true = cdr.view(-1).float()

        L_CDR = F.mse_loss(cdr_pred, cdr_true)

        feat = out["bottleneck"]
        feat = F.adaptive_avg_pool3d(feat, (2, 2, 2))
        feat = feat.view(feat.size(0), -1)

        L_contrast = info_nce_loss(feat, temperature=0.1)

        loss = (
            2.5 *L_rec.mean()
            + beta * L_KL.mean()
            + L_CDR.mean()
            + gamma * L_contrast
        )

        loss = loss.mean()

        if not torch.isfinite(loss):
            print(" NaN/Inf loss detected — skipping batch")
            optimizer.zero_grad(set_to_none=True)
            continue   


        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        optimizer.step()

        total_loss += loss.item()
        total_rec += L_rec.item()
        total_kl += L_KL.item()
        total_cdr += L_CDR.item()
        total_contrast += L_contrast.item()

        valid_batches += 1

    if valid_batches == 0:
        return {
            "loss": float("nan"),
            "recon": float("nan"),
            "kl": float("nan"),
            "cdr": float("nan"),
            "contrast": float("nan")
        }

    return {
        "loss": total_loss / valid_batches,
        "recon": total_rec / valid_batches,
        "kl": total_kl / valid_batches,
        "cdr": total_cdr / valid_batches,
        "contrast": total_contrast / valid_batches
    }

    
def train(model, train_loader, epochs=50, lr=1e-4):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        metrics = train_one_epoch(model, train_loader, optimizer)

        print(metrics)

        history.append(metrics)

    return history

In [ ]:
# Model initialization

# T1-weighted 
model_t1w = VAE(latent_dim=64).to(device)

# T2-weighted 
model_t2w = VAE(latent_dim=64).to(device)

In [ ]:
# Training runs

# train VAE on T1-weighted data
losses_t1w = train(model_t1w, train_loader_t1w, epochs=50, lr=1e-4)

# train VAE on T2-weighted data
losses_t2w = train(model_t2w, train_loader_t2w, epochs=50, lr=1e-4)

In [ ]:
# Save Models and testing data loaders

# save T1-weighted model
torch.save(model_t1w.state_dict(), "model_t1w.pt")

# save T2-weighted model
torch.save(model_t2w.state_dict(), "model_t2w.pt")

In [1]:
# Evaluation function


def ordinal_auc(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    concordant = 0
    discordant = 0

    n = len(y_true)

    for i in range(n):
        for j in range(i + 1, n):

            if y_true[i] == y_true[j]:
                continue

            if (y_true[i] - y_true[j]) * (y_score[i] - y_score[j]) > 0:
                concordant += 1
            else:
                discordant += 1

    return concordant / (concordant + discordant + 1e-8)


def fixed_ovr_metrics(y_true_cls, y_score_cont):

    classes = np.unique(y_true_cls)

    roc_aucs = []
    pr_aucs = []

    print("\n========== OVR ROC-AUC / PR-AUC ==========")

    for c in classes:

        y_bin = (y_true_cls == c).astype(int)

        if c == 0:
            score = -y_score_cont  
        else:
            score = y_score_cont

        # ROC
        fpr, tpr, _ = roc_curve(y_bin, score)
        roc_auc = auc(fpr, tpr)

        # PR
        precision, recall, _ = precision_recall_curve(y_bin, score)
        pr_auc = auc(recall, precision)

        roc_aucs.append(roc_auc)
        pr_aucs.append(pr_auc)

        print(f"Class {c}: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}")

    print(f"\nMacro ROC-AUC: {np.mean(roc_aucs):.4f}")
    print(f"Macro PR-AUC:  {np.mean(pr_aucs):.4f}")

    return roc_aucs, pr_aucs


def full_evaluation(model, test_loader):

    model.eval()

    y_true = []
    y_score = []

    with torch.no_grad():
        for batch in test_loader:

            x, age, sex, cdr, group = batch

            x = x.to(device)
            age = age.to(device)
            sex = sex.to(device)

            out = model(x, age, sex)

            pred = out["cdr_pred"].squeeze(-1).cpu().numpy()

            y_true.extend(cdr.cpu().numpy())
            y_score.extend(pred)

    y_true = np.array(y_true)
    y_score = np.array(y_score)

    y_true_cls = np.array([severity_group(x) for x in y_true])
    y_pred_cls = np.array([severity_group(x) for x in y_score])

    acc = accuracy_score(y_true_cls, y_pred_cls)
    f1_w = f1_score(y_true_cls, y_pred_cls, average="weighted")
    f1_m = f1_score(y_true_cls, y_pred_cls, average="macro")

    prec_m = precision_score(y_true_cls, y_pred_cls, average="macro")
    rec_m = recall_score(y_true_cls, y_pred_cls, average="macro")

    bal_acc = balanced_accuracy_score(y_true_cls, y_pred_cls)

    print("\n================ CLASSIFICATION =================")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 weighted: {f1_w:.4f}")
    print(f"F1 macro: {f1_m:.4f}")
    print(f"Precision macro: {prec_m:.4f}")
    print(f"Recall macro: {rec_m:.4f}")
    print(f"Balanced Acc: {bal_acc:.4f}")

    print(classification_report(y_true_cls, y_pred_cls))

    cm = confusion_matrix(y_true_cls, y_pred_cls)

    print("\nConfusion Matrix:\n", cm)

    print("\n================ SENS / SPEC =================")

    for i in range(len(cm)):
        TP = cm[i, i]
        FN = cm[i].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        sens = TP / (TP + FN + 1e-8)
        spec = TN / (TN + FP + 1e-8)

        print(f"Class {i}: Sens={sens:.3f}, Spec={spec:.3f}")

    kappa = cohen_kappa_score(y_true_cls, y_pred_cls)
    qkappa = cohen_kappa_score(y_true_cls, y_pred_cls, weights="quadratic")

    print("\n================ AGREEMENT =================")
    print(f"Kappa: {kappa:.4f}")
    print(f"Quadratic Kappa: {qkappa:.4f}")

    ord_auc = ordinal_auc(y_true_cls, y_score)

    print("\n================ ORDINAL AUC =================")
    print(f"Ordinal AUC: {ord_auc:.4f}")

    overall_mean = np.mean(y_score)

    between, within = 0, 0

    for g in np.unique(y_true_cls):
        vals = y_score[y_true_cls == g]
        m = np.mean(vals)

        between += len(vals) * (m - overall_mean) ** 2
        within += np.var(vals) * len(vals)

    sep = between / (within + 1e-8)

    print("\n================ SEPARATION =================")
    print(f"Separation: {sep:.4f}")

    roc_aucs, pr_aucs = fixed_ovr_metrics(y_true_cls, y_score)

    return {
        "acc": acc,
        "f1_macro": f1_m,
        "f1_weighted": f1_w,
        "kappa": kappa,
        "kappa_q": qkappa,
        "ordinal_auc": ord_auc,
        "roc_auc_macro": np.mean(roc_aucs),
        "pr_auc_macro": np.mean(pr_aucs),
        "separation": sep
    }

In [ ]:
# Run evaluations

# T1w evaluation
results_t1w = full_evaluation(model_t1w, test_loaderr_t1w)

# T2w evaluation
results_t2w = full_evaluation(model_t2w, test_loaderr_t2w)

In [ ]:
# Brain reconstruction visualization of middle slice
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def visualize_reconstruction(model, dataloader, num_samples=3):

    model.eval()

    x, age, sex, cdr, groups = next(iter(dataloader))

    x = x.to(device)
    x = torch.clamp(x, min=0)
    x = x / (x.max() + 1e-5)    
    age = age.to(device)
    sex = sex.to(device)

    with torch.no_grad():
        out = model(x, age, sex)

    recon = out["recon"]
    cdr_pred = out["cdr_pred"].squeeze(-1)

    x = x.cpu()
    recon = recon.cpu()
    cdr = cdr.cpu()
    cdr_pred = cdr_pred.cpu()
    groups = groups.cpu()

    num_samples = min(num_samples, x.size(0))

    for i in range(num_samples):

        mid = x.shape[2] // 2

        real_slice = x[i, 0, mid].numpy()
        recon_slice = recon[i, 0, mid].numpy()

        true_cdr = float(cdr[i])
        pred_cdr = float(cdr_pred[i])
        true_group = int(groups[i])

        plt.figure(figsize=(8, 4))

        plt.subplot(1, 2, 1)
        plt.imshow(real_slice, cmap="gray")
        plt.title(f"REAL\nCDR: {true_cdr:.2f}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(recon_slice, cmap="gray")
        plt.title(f"RECON\nPred CDR: {pred_cdr:.2f}")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        print("=" * 50)
        print(f"Sample {i}")
        print(f"True CDR   : {true_cdr:.3f}")
        print(f"Pred CDR   : {pred_cdr:.3f}")
        print("=" * 50)